# LSTM-based Reduced-Time Recursive Physics-Informed Neural Network for Shape Memory Materials

## Stress-Relaxation Inverse Characterization

This notebook performs inverse identification for the 45° off-axis stress-relaxation test using an LSTM-based RT-RPINN framework.


## Import Required Libraries


In [ ]:
import sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR  # noqa: F401 (used in train)
import matplotlib.pyplot as plt
from pathlib import Path
import time
import math


class _Tee:
    """Mirror writes to both an original stream and a file."""
    def __init__(self, original, file_path):
        self._orig = original
        self._file = open(file_path, 'w', buffering=1, encoding='utf-8')

    def write(self, data):
        self._orig.write(data)
        self._file.write(data)

    def flush(self):
        self._orig.flush()
        self._file.flush()

    def close(self):
        self._file.close()

    # Proxy any other attribute access (e.g. .fileno, .isatty) to the original
    def __getattr__(self, name):
        return getattr(self._orig, name)

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


# 1. Data Loader for FE Results


## FEDataLoader


In [ ]:
class FEDataLoader:
    """Load and process FE simulation results from CSV files"""

    def __init__(self, data_dir, rf_file, u_file, frame_time_file):
        self.data_dir = Path(data_dir)
        self.rf_file = Path(rf_file)
        self.u_file = Path(u_file)
        self.frame_time_file = Path(frame_time_file)
        self.data = []
        self.frame_time_map = {}

    def load_all_data(self):
        """Load all CSV files from the results directory"""
        import re

        # Load frame-time mapping (REQUIRED for correct time assignment with Abaqus adaptive stepping)
        frame_time_df = pd.read_csv(self.frame_time_file, header=None, names=['Frame', 'Time'])
        # Create dictionary: frame_idx -> actual_time
        self.frame_time_map = dict(zip(frame_time_df['Frame'].values,
                                      frame_time_df['Time'].values))

        # Helper function: robust frame index extraction using regex
        def frame_from_filename(filepath):
            """Extract frame index from filename using regex (more robust than split)"""
            m = re.search(r'frame(\d+)', filepath.stem)
            if not m:
                raise ValueError(f"Cannot parse frame index from filename: {filepath.name}")
            return int(m.group(1))

        # Load full-field data
        step_files = sorted(
            list(self.data_dir.glob("*_Step-loading_frame*.csv")),
            key=frame_from_filename
        )

        # Sanity check: Verify all data frames are covered by frame_time_map
        data_frames = [frame_from_filename(f) for f in step_files]
        missing_frames = [f for f in data_frames if f not in self.frame_time_map]
        if missing_frames:
            raise ValueError(
                f"Frame-time mapping incomplete! Missing frames: {missing_frames}. "
                f"frame_time_file must cover all data frames."
            )

        for csv_file in step_files:
            df = pd.read_csv(csv_file)

            # Extract frame number from filename (using regex helper)
            frame_idx = frame_from_filename(csv_file)

            # Get actual time from frame-time map (with explicit error handling)
            frame_time = self.frame_time_map.get(frame_idx, None)
            if frame_time is None:
                raise KeyError(
                    f"Frame {frame_idx} not found in frame_time_map. "
                    f"Available frames: {sorted(self.frame_time_map.keys())}. "
                    f"Check frame_time_file consistency with data files."
                )

            df['Time'] = frame_time
            df['Frame'] = frame_idx
            # Constant reference temperature (isothermal test at 323K)
            # Since T = T_ref throughout, reduced time = real time (no WLF shift)
            df['Temperature'] = 323.0

            self.data.append(df)

        # Concatenate all data
        self.full_data = pd.concat(self.data, ignore_index=True)

        # Load reaction force and displacement history
        # BEST PRACTICE: If possible, export RF/U files with Frame column directly
        # (e.g., [Frame, RF] instead of [Time, RF]) to completely avoid float matching
        rf_df = pd.read_csv(self.rf_file, header=None, names=['Time', 'RF'])
        u_df = pd.read_csv(self.u_file, header=None, names=['Time', 'U'])

        # CRITICAL FIX: Add Frame column to RF and U data using frame_time_map
        # This ensures strict Frame-based alignment instead of error-prone float Time matching
        # Vectorized matching with tolerance check (O(N log M) instead of O(N*M))

        # Prepare sorted arrays for efficient binary search
        frame_ids = np.array(list(self.frame_time_map.keys()), dtype=int)
        frame_times = np.array([self.frame_time_map[f] for f in frame_ids], dtype=float)

        # Sort by time
        order = np.argsort(frame_times)
        frame_ids = frame_ids[order]
        frame_times = frame_times[order]

        def match_time_to_frame(times, tol_time=1e-5):
            """Vectorized time-to-frame matching with tolerance check"""
            times = np.asarray(times, dtype=float)
            # Binary search for insertion points
            idx = np.searchsorted(frame_times, times)
            idx = np.clip(idx, 1, len(frame_times) - 1)
            left = idx - 1
            right = idx
            # Choose closer neighbor
            choose_right = (np.abs(frame_times[right] - times) < np.abs(frame_times[left] - times))
            best = np.where(choose_right, right, left)
            best_frames = frame_ids[best]

            # Tolerance check: expose silent mismatches
            err = np.abs(frame_times[best] - times)
            if np.any(err > tol_time):
                n_bad = np.sum(err > tol_time)
                max_err = err.max()
                print(f"  [Warning] {n_bad} points exceed tol_time={tol_time:.2e}. Max err={max_err:.3e}")
                print(f"  This may indicate RF/U time misalignment with frame_time_map.")

            return best_frames

        # Match RF and U data to frames
        rf_df['Frame'] = match_time_to_frame(rf_df['Time'].values, tol_time=1e-5)
        u_df['Frame'] = match_time_to_frame(u_df['Time'].values, tol_time=1e-5)

        self.rf_data = rf_df
        self.u_data = u_df

        print(f"\nFrame-Time alignment summary:")
        print(f"  Full field data: {len(self.full_data['Frame'].unique())} frames")
        print(f"  RF data: {len(self.rf_data)} points, Frames {self.rf_data['Frame'].min()}-{self.rf_data['Frame'].max()}")
        print(f"  U data: {len(self.u_data)} points, Frames {self.u_data['Frame'].min()}-{self.u_data['Frame'].max()}")

        return self.full_data

    def get_domain_bounds(self):
        """Get the spatial and temporal domain bounds"""
        bounds = {
            'x_min': self.full_data['X'].min(),
            'x_max': self.full_data['X'].max(),
            'y_min': self.full_data['Y'].min(),
            'y_max': self.full_data['Y'].max(),
            'z_min': self.full_data['Z'].min(),
            'z_max': self.full_data['Z'].max(),
            't_min': self.full_data['Time'].min(),
            't_max': self.full_data['Time'].max(),
        }
        return bounds

    def get_boundary_node_ids(self, face='left', tol=1e-4):
        """
        Extract boundary node IDs for consistent cross-time alignment.

        Returns NodeID set instead of full data to avoid frame-mixing issues.
        Use this to ensure same spatial points across different time steps.

        Args:
            face: 'left' (x_min) or 'right' (x_max)
            tol: Spatial tolerance (mm). Default 1e-4 mm (conservative for Abaqus output)

        Returns:
            Tuple: (node_ids, node_column_name)
        """
        # Detect node ID column (Abaqus may use Node, NodeLabel, or NID)
        for col in ['NodeLabel', 'Node', 'NID']:
            if col in self.full_data.columns:
                node_col = col
                break
        else:
            raise KeyError("No node ID column found (expected NodeLabel/Node/NID).")

        # Get boundary coordinate
        if face == 'left':
            x_boundary = self.full_data['X'].min()
        elif face == 'right':
            x_boundary = self.full_data['X'].max()
        else:
            raise ValueError(f"Invalid face: {face}. Must be 'left' or 'right'.")

        # Extract nodes on boundary (across all frames)
        boundary_data = self.full_data[np.abs(self.full_data['X'] - x_boundary) < tol]
        node_ids = np.sort(boundary_data[node_col].unique())

        print(f"  {face.capitalize()} boundary: {len(node_ids)} unique nodes (tol={tol:.2e} mm)")

        return node_ids, node_col

# 2. Trainable Material Parameters


## Model and Material Parameters


In [ ]:
class InverseMaterialParams(nn.Module):
    """
    Material parameters for Scheme B inverse identification.

    Trainable:
    - C11(0), C22(0), C66(0)
    - A_g: total matrix relaxation amplitude for g2/g3/g4

    Fixed:
    - C12(0), C23(0)
    - g0, g1
    - relative shape of g2:g3:g4
    - fiber spectrum for C11 (elastic)

    Stress form: σ = C^∞ : ε + Σ_k C^(k) : (ε - q^(k))
    """

    def __init__(self):
        super().__init__()

        # 5 branches matching the target Prony series (tau=[0.1,1,10,100,1000]s).
        # The τ=10000s branch was removed: it behaves as a quasi-equilibrium
        # modulus within the 900s experiment (exp(-900/10000)=91%), making it
        # indistinguishable from g_inf and causing g_inf→0 in the inverse solution.
        self.N_prony = 5
        self.register_buffer("rho", torch.tensor([0.1, 1.0, 10.0, 100.0, 1000.0], dtype=torch.float32))

        # ===== Total stiffness C(0) components =====
        # Scheme B: only C11, C22, and C66 are trainable.
        self.logC11_total = nn.Parameter(torch.tensor(np.log(5000.0),  dtype=torch.float32))  # True: 11250
        self.logC22_total = nn.Parameter(torch.tensor(np.log(700.0),   dtype=torch.float32))  # True: 1426
        self.logC66_total = nn.Parameter(torch.tensor(np.log(135.0),   dtype=torch.float32))  # True: 268
        self.register_buffer("logC12_total_fixed", torch.tensor(np.log(891.26), dtype=torch.float32))
        self.register_buffer("logC23_total_fixed", torch.tensor(np.log(915.56), dtype=torch.float32))
        self.register_buffer("C11_ref", torch.tensor(11250.16, dtype=torch.float32))
        self.register_buffer("C22_ref", torch.tensor(1425.85, dtype=torch.float32))
        self.register_buffer("C66_ref", torch.tensor(267.69, dtype=torch.float32))

        # ===== Reduced matrix spectrum parameterization =====
        self.g0_fixed = 0.206  # τ=0.1s: fixed at UMAT true value
        self.g1_fixed = 0.093  # τ=1.0s: fixed at UMAT true value
        self._g_free_sum = 1.0 - self.g0_fixed - self.g1_fixed  # = 0.701
        self.g_inf_min = 0.001
        self.Ag_max = self._g_free_sum - self.g_inf_min
        self.register_buffer(
            "g_rel_shape",
            torch.tensor([0.306, 0.358, 0.034], dtype=torch.float32) / (0.306 + 0.358 + 0.034)
        )
        self.register_buffer("Ag_ref", torch.tensor(0.306 + 0.358 + 0.034, dtype=torch.float32))
        Ag_init = 0.55
        init_ratio = min(max(Ag_init / self.Ag_max, 1e-4), 1.0 - 1e-4)
        self.logit_Ag = nn.Parameter(torch.tensor(np.log(init_ratio / (1.0 - init_ratio)), dtype=torch.float32))

        self.T_ref = 323.0  # Reference temperature (K)

        # 45-degree rotation matrix (Material -> Global)
        theta = np.deg2rad(45.0)
        c, s = np.cos(theta), np.sin(theta)
        self.register_buffer("Q", torch.tensor([
            [ c, -s,  0],
            [ s,  c,  0],
            [ 0,  0,  1]
        ], dtype=torch.float32))

    # ===== Properties: Spectrum weights =====
    # Group 1 (FIBER): C11 - nearly elastic (~8% relaxation)
    # Group 2 (MATRIX): C12, C22, C23, C66 - highly viscoelastic (~99% relaxation)

    @property
    def g_fiber(self):
        """
        Fiber group (C11) Spectrum: FORCED ELASTIC
        Carbon fiber does NOT relax significantly: g_k = 0, g_inf = 1.0
        True relaxation is ~8%, but we approximate as elastic for simplicity.
        """
        dev = self.rho.device  # Use buffer device reference
        g = torch.zeros(self.N_prony + 1, device=dev, dtype=torch.float32)
        g[-1] = 1.0  # g_inf = 1 (pure elastic)
        return g

    @property
    def g_matrix(self):
        """Matrix spectrum with one trainable relaxation-amplitude parameter A_g."""
        dev = self.rho.device
        g_fixed = torch.tensor([self.g0_fixed, self.g1_fixed], dtype=torch.float32, device=dev)
        Ag = torch.sigmoid(self.logit_Ag) * self.Ag_max
        g_rel = Ag * self.g_rel_shape.to(dev)
        g_inf = self._g_free_sum - Ag
        return torch.cat([g_fixed, g_rel, g_inf.view(1)], dim=0)

    @property
    def g_C11(self):
        """C11 Spectrum: Fiber-dominated -> elastic"""
        return self.g_fiber

    @property
    def g_C12(self):
        """C12 Spectrum: Matrix-dominated (highly viscoelastic) -> use g_matrix"""
        return self.g_matrix

    @property
    def g_C22(self):
        """C22 Spectrum: Matrix-dominated -> use g_matrix"""
        return self.g_matrix

    @property
    def g_C23(self):
        """C23 Spectrum: Matrix-dominated -> use g_matrix"""
        return self.g_matrix

    @property
    def g_C66(self):
        """C66 Spectrum: Matrix-dominated -> use g_matrix"""
        return self.g_matrix

    # ===== Properties: Total stiffness C(0) =====

    @property
    def C11_total(self):
        """Total instantaneous stiffness C11(0)"""
        return torch.exp(self.logC11_total)

    @property
    def C12_total(self):
        """Total instantaneous stiffness C12(0)"""
        return torch.exp(self.logC12_total_fixed)

    @property
    def C22_total(self):
        """Total instantaneous stiffness C22(0)"""
        return torch.exp(self.logC22_total)

    @property
    def C23_total(self):
        """Total instantaneous stiffness C23(0), fixed in Scheme B."""
        return torch.exp(self.logC23_total_fixed)

    @property
    def C66_total(self):
        """Total instantaneous stiffness C66(0)"""
        return torch.exp(self.logC66_total)

    # ===== Properties: Long-term stiffness C^∞ (DERIVED from softmax) =====

    @property
    def C11_inf(self):
        """Long-term stiffness C11^∞ = g_inf^C11 * C11(0)"""
        return self.g_C11[-1] * self.C11_total

    @property
    def C12_inf(self):
        """Long-term stiffness C12^∞ = g_inf^C12 * C12(0)"""
        return self.g_C12[-1] * self.C12_total

    @property
    def C22_inf(self):
        """Long-term stiffness C22^∞ = g_inf^C22 * C22(0)"""
        return self.g_C22[-1] * self.C22_total

    @property
    def C23_inf(self):
        """Long-term stiffness C23^∞ = g_inf^C23 * C23(0)"""
        return self.g_C23[-1] * self.C23_total

    @property
    def C44_inf(self):
        """Long-term shear C44^∞ = (C22^∞ - C23^∞)/2"""
        return (self.C22_inf - self.C23_inf) / 2.0

    @property
    def C66_inf(self):
        """Long-term shear C66^∞ = g_inf^C66 * C66(0)"""
        return self.g_C66[-1] * self.C66_total

    # ===== Properties: Per-branch relaxation strengths ΔC^(k) (DERIVED from softmax) =====

    @property
    def dC11_k(self):
        """Per-branch relaxation strength ΔC11^(k) = g_k^C11 * C11(0), k=0...5

        CRITICAL: Uses softmax weights, so automatically positive and well-distributed.
        No need for softplus or manual normalization!
        """
        return self.g_C11[:-1] * self.C11_total  # (6,) - exclude g_inf

    @property
    def dC12_k(self):
        """Per-branch relaxation strength ΔC12^(k) = g_k^C12 * C12(0), k=0...5"""
        return self.g_C12[:-1] * self.C12_total  # (6,)

    @property
    def dC22_k(self):
        """Per-branch relaxation strength ΔC22^(k) = g_k^C22 * C22(0), k=0...5"""
        return self.g_C22[:-1] * self.C22_total  # (6,)

    @property
    def dC23_k(self):
        """Per-branch relaxation strength ΔC23^(k) = g_k^C23 * C23(0), k=0...5

        CRITICAL: C44_k = (C22_k - C23_k)/2 stability is ensured by:
        - C23(0) < C22(0) from ratio parameterization
        - g_k^C23 and g_k^C22 are independent (can have different spectra)
        """
        return self.g_C23[:-1] * self.C23_total  # (6,)

    @property
    def dC66_k(self):
        """Per-branch relaxation strength ΔC66^(k) = g_k^C66 * C66(0), k=0...5"""
        return self.g_C66[:-1] * self.C66_total  # (6,)

    # ===== Properties: Total relaxation strengths ΔC = Σ_k ΔC^(k) =====

    @property
    def dC11(self):
        """Total relaxation strength ΔC11 = Σ_k ΔC11^(k) > 0"""
        return self.dC11_k.sum()

    @property
    def dC12(self):
        """Total relaxation strength ΔC12 = Σ_k ΔC12^(k) > 0"""
        return self.dC12_k.sum()

    @property
    def dC22(self):
        """Total relaxation strength ΔC22 = Σ_k ΔC22^(k) > 0"""
        return self.dC22_k.sum()

    @property
    def dC23(self):
        """Total relaxation strength ΔC23 = Σ_k ΔC23^(k) > 0"""
        return self.dC23_k.sum()

    @property
    def dC66(self):
        """Total relaxation strength ΔC66 = Σ_k ΔC66^(k) > 0"""
        return self.dC66_k.sum()

    # ===== Properties: Instantaneous stiffness C(0) = C^∞ + ΔC =====

    @property
    def C11_inst(self):
        """Instantaneous stiffness C11(0) = C11^∞ + ΔC11"""
        return self.C11_inf + self.dC11

    @property
    def C12_inst(self):
        """Instantaneous stiffness C12(0) = C12^∞ + ΔC12"""
        return self.C12_inf + self.dC12

    @property
    def C22_inst(self):
        """Instantaneous stiffness C22(0) = C22^∞ + ΔC22"""
        return self.C22_inf + self.dC22

    @property
    def C23_inst(self):
        """Instantaneous stiffness C23(0) = C23^∞ + ΔC23

        CRITICAL FIX: Use self.dC23 (already computed as sum of dC23_k via ratio)
        """
        return self.C23_inf + self.dC23

    @property
    def C44_inst(self):
        """Instantaneous shear C44(0) = (C22(0) - C23(0))/2"""
        return (self.C22_inst - self.C23_inst) / 2.0

    @property
    def C66_inst(self):
        """Instantaneous shear C66(0) = C66^∞ + ΔC66"""
        return self.C66_inf + self.dC66

    def get_spectrum_weights(self):
        """
        Return MATRIX spectrum weights (the learned viscoelastic spectrum).

        Scheme B model:
        - C11 (Fiber): FROZEN at g_inf=1 (elastic), NOT returned here
        - Matrix (C12, C22, C23, C66): Learned spectrum returned here

        Returns: (N_prony + 1,) tensor [g0, g1, g2, g3, g4, g_inf] for matrix
        """
        return self.g_matrix

    def get_C_branch(self, k):
        """
        Get Prony branch k stiffness components - DIRECTLY from ΔC^(k).

        In Scheme B:
        - C11 fiber branches are zero (elastic fiber)
        - C12, C22, C23, C66 share the same reduced matrix spectrum
        - branch magnitudes differ because the underlying stiffness components differ

        Args:
            k: Branch index (0 to N_prony-1)

        Returns:
            C11_k, C12_k, C22_k, C23_k, C66_k: Branch k moduli
        """
        # DIRECT access to branch k (NO fraction multiplier)
        C11_k = self.dC11_k[k]
        C12_k = self.dC12_k[k]
        C22_k = self.dC22_k[k]
        C23_k = self.dC23_k[k]
        C66_k = self.dC66_k[k]

        return C11_k, C12_k, C22_k, C23_k, C66_k

    # DELETED: get_stiffness_matrix_inst() - NOT USED in component-wise Prony model
    # Component-wise stress form: σ = C^∞ : ε + Σ_k C^(k) : (ε - q^(k))
    # Only need: C^∞ (below) and C^(k) (get_stiffness_matrix_branch)
    # C(0) properties are kept for reference/validation only (C11_inst, C22_inst, etc.)

    def get_stiffness_matrix_inf(self):
        """
        Construct long-term stiffness matrix C^∞ (component-wise Prony model).

        Used in stress computation: σ = C^∞ : ε + Σ_k C^(k) : (ε - q^(k))
        This provides the equilibrium (infinite-time) elastic response.

        Transverse isotropy constraints:
        - C33 = C22, C13 = C12 (fiber symmetry)
        - C44 = (C22 - C23) / 2 (UMAT coupling)
        - C55 = C66 (in-plane/out-of-plane shear)

        Returns: 6×6 stiffness matrix (Voigt notation)
        """
        C11 = self.C11_inf
        C12 = self.C12_inf
        C22 = self.C22_inf
        C23 = self.C23_inf
        C33 = C22  # Transverse isotropy: C33 = C22
        C13 = C12  # Transverse isotropy: C13 = C12
        C44 = self.C44_inf  # = (C22 - C23) / 2
        C55 = self.C66_inf
        C66 = self.C66_inf

        # CRITICAL FIX: Use self's device, not global device variable
        dev = self.logC22_total.device
        C = torch.zeros(6, 6, device=dev, dtype=torch.float32)

        # Normal components
        C[0, 0] = C11
        C[1, 1] = C22
        C[2, 2] = C33

        # Coupling (symmetric)
        C[0, 1] = C12; C[1, 0] = C12
        C[0, 2] = C13; C[2, 0] = C13
        C[1, 2] = C23; C[2, 1] = C23

        # CRITICAL FIX: Shear components must match Voigt ordering [11,22,33,12,13,23]
        # Current PDE/strain uses (12,13,23) order for indices (3,4,5)
        C[3, 3] = C66  # Index 3 → 12 component (in-plane shear)
        C[4, 4] = C55  # Index 4 → 13 component (out-of-plane shear, = C66 by symmetry)
        C[5, 5] = C44  # Index 5 → 23 component (= (C22-C23)/2, transverse shear)

        return C

    def get_stiffness_matrix_branch(self, k):
        """
        Construct Prony branch k stiffness matrix C^(k) (component-wise model).

        Used in stress computation: σ = C^∞ : ε + Σ_k C^(k) : (ε - q^(k))
        Under Scheme B, the matrix components share one reduced spectrum and
        the weakly sensitive elastic components C12 and C23 are fixed.

        Args:
            k: Branch index (0 to N_prony-1)

        Returns: 6×6 stiffness matrix for branch k (Voigt notation)
        """
        C11_k, C12_k, C22_k, C23_k, C66_k = self.get_C_branch(k)

        C33_k = C22_k  # Transverse isotropy
        C13_k = C12_k  # Transverse isotropy
        C44_k = (C22_k - C23_k) / 2.0  # UMAT coupling (guaranteed > 0 via ratio)
        C55_k = C66_k

        # CRITICAL FIX: Use self's device, not global device variable
        dev = self.logC22_total.device
        C = torch.zeros(6, 6, device=dev, dtype=torch.float32)

        C[0, 0] = C11_k
        C[1, 1] = C22_k
        C[2, 2] = C33_k

        C[0, 1] = C12_k; C[1, 0] = C12_k
        C[0, 2] = C13_k; C[2, 0] = C13_k
        C[1, 2] = C23_k; C[2, 1] = C23_k

        # CRITICAL FIX: Shear components must match Voigt ordering [11,22,33,12,13,23]
        # Current PDE/strain uses (12,13,23) order for indices (3,4,5)
        C[3, 3] = C66_k  # Index 3 → 12 component (in-plane shear)
        C[4, 4] = C55_k  # Index 4 → 13 component (out-of-plane shear, = C66_k by symmetry)
        C[5, 5] = C44_k  # Index 5 → 23 component (= (C22_k-C23_k)/2, transverse shear)

        return C

    def compute_parameter_penalty(self):
        """
        Prior regularization for Scheme B.

        This penalty keeps the low-dimensional inverse problem in a physically
        meaningful region and suppresses the common collapse mode:
        low stiffness + overly fast relaxation.
        """
        dev = self.rho.device  # Use buffer device reference
        penalty = torch.zeros((), device=dev, dtype=torch.float32)

        c11_rel = (self.C11_total - self.C11_ref) / self.C11_ref
        c22_rel = (self.C22_total - self.C22_ref) / self.C22_ref
        c66_rel = (self.C66_total - self.C66_ref) / self.C66_ref
        Ag = torch.sigmoid(self.logit_Ag) * self.Ag_max
        Ag_rel = (Ag - self.Ag_ref) / self.Ag_ref

        penalty = penalty + 0.05 * c11_rel.pow(2)
        penalty = penalty + 0.05 * c22_rel.pow(2)
        penalty = penalty + 0.08 * c66_rel.pow(2)
        penalty = penalty + 0.08 * Ag_rel.pow(2)

        return penalty


# 3. Physics-Informed Neural Network

class LSTMInversePINN(nn.Module):
    """
    LSTM-PINN for inverse material characterization.

    Architecture:
    1. Input normalization + fiber coordinate encoding (6D input)
    2. Spatial feature extraction (MLP)
    3. LSTM for temporal sequence processing
    4. Output decoder for displacement prediction

    Args:
        spatial_hidden: Hidden layer sizes for spatial encoder
        lstm_hidden: LSTM hidden state dimension
        lstm_layers: Number of LSTM layers
        dropout: Dropout probability for regularization
    """

    def __init__(
        self,
        spatial_hidden=(64, 64),
        lstm_hidden=64,
        lstm_layers=1,
        dropout=0.0,
    ):
        super().__init__()

        self.lstm_hidden = lstm_hidden
        self.lstm_layers = lstm_layers

        # Register rotation buffers for 45° fiber orientation
        c = math.cos(math.radians(45.0))
        s = math.sin(math.radians(45.0))
        self.register_buffer("cos_theta", torch.tensor(c, dtype=torch.float32))
        self.register_buffer("sin_theta", torch.tensor(s, dtype=torch.float32))

        # Spatial encoder: 6D (x, y, z, t, x_local, y_local) -> features
        dims = [6, *spatial_hidden]
        mlp = []
        for i in range(len(dims) - 1):
            mlp.append(nn.Linear(dims[i], dims[i + 1]))
            mlp.append(nn.Tanh())
            if dropout > 0:
                mlp.append(nn.Dropout(dropout))
        self.spatial_encoder = nn.Sequential(*mlp)

        # LSTM for temporal processing (batch_first=True)
        self.lstm = nn.LSTM(
            input_size=dims[-1],
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            dropout=dropout if lstm_layers > 1 else 0.0,
        )

        # Residual temporal head: LSTM hidden -> displacement correction
        self.temporal_decoder = nn.Sequential(
            nn.Linear(lstm_hidden, lstm_hidden),
            nn.Tanh(),
            nn.Linear(lstm_hidden, 3),
        )
        # Spatial baseline head: direct mapping from spatial features to displacement.
        # Final output = spatial baseline + temporal residual.
        self.spatial_head = nn.Linear(dims[-1], 3)

        self._init_weights()

    def _init_weights(self):
        """Xavier initialization for better convergence"""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def _encode_spatial(self, x, y, z, t):
        """
        Encode spatial-temporal coordinates with fiber-aligned features.

        Args:
            x, y, z, t: Normalized coordinates (batch, seq, 1) or (batch, 1)

        Returns:
            Spatial features (batch, seq, feature_dim) or (batch, feature_dim)
        """
        # Fiber coordinate transformation (45° rotation)
        x_local = x * self.cos_theta + y * self.sin_theta
        y_local = -x * self.sin_theta + y * self.cos_theta
        h = torch.cat([x, y, z, t, x_local, y_local], dim=-1)
        return self.spatial_encoder(h)

    def forward_sequence(self, x, y, z, t, h0=None, c0=None):
        """
        Forward pass for temporal sequences.

        Args:
            x, y, z, t: Normalized coordinates, shape (batch, seq, 1)
            h0, c0: Optional initial LSTM hidden states

        Returns:
            u_seq: Displacement sequence (batch, seq, 3)
            (h, c): Final LSTM hidden states
        """
        # Encode spatial features for entire sequence
        spatial_feats = self._encode_spatial(x, y, z, t)  # (B, S, F)

        # LSTM processing
        lstm_out, (h, c) = self.lstm(
            spatial_feats,
            (h0, c0) if h0 is not None else None
        )  # (B, S, lstm_hidden)

        # Easier optimization: a direct spatial baseline plus temporal correction.
        u_spatial = self.spatial_head(spatial_feats)      # (B, S, 3)
        u_temporal = self.temporal_decoder(lstm_out)      # (B, S, 3)
        u_seq = u_spatial + u_temporal

        return u_seq, (h, c)





## Solver and Loss Functions


In [ ]:
class InversePINNSolver:
    """
    Solver for inverse material parameter identification using PINN.

    Simultaneously optimizes:
    - Neural network weights (to fit displacement field u)
    - Material parameters (stiffness C, spectrum weights g)
    """

    @staticmethod
    def compute_q_recursive(strain_seq, dt_seq, rho_array):
        """
        Internal-variable recursion aligned with EX1/EX3.

        Definition of q here:
        - q is a branch strain-like internal variable.
        - Recurrence:
          q^(k)_n = exp(-Δt/ρ_k) * q^(k)_{n-1}
                    + (1 - exp(-Δt/ρ_k)) * ε_n
        - During a hold, q approaches ε and the branch stress term
          C^(k):(ε-q^(k)) decays to zero.
        - Temperature shift omitted (isothermal).

        Args:
            strain_seq: (N_t, N_pts, 6) - total strain history (engineering Voigt)
            dt_seq: (N_t-1,) time increments
            rho_array: (N_prony,) relaxation times

        Returns:
            q_seq: (N_t, N_pts, 6, N_prony) history variables per branch
        """
        N_t, N_pts, _ = strain_seq.shape
        N_prony = len(rho_array)
        device = strain_seq.device

        q_seq = torch.zeros(N_t, N_pts, 6, N_prony, device=device, dtype=strain_seq.dtype)

        # IC: q(t=0) = 0
        q_seq[0, :, :, :] = 0.0

        # Time-stepping
        # GRADIENT TRUNCATION: detach every 50 steps to limit memory
        truncate_every = 50

        rho_broadcast = rho_array.view(1, 1, N_prony)  # (1, 1, N_prony)

        for n in range(1, N_t):
            dt = dt_seq[n-1]
            strain_n = strain_seq[n, :, :]  # (N_pts, 6)

            dtr_over_rho = dt / rho_broadcast  # (1, 1, N_prony)
            exp_factor = torch.exp(-dtr_over_rho)  # (1, 1, N_prony)
            relax_factor = 1.0 - exp_factor

            q_prev = q_seq[n-1, :, :, :]  # (N_pts, 6, N_prony)
            if n % truncate_every == 0:
                q_prev = q_prev.detach()

            q_seq[n, :, :, :] = exp_factor * q_prev + strain_n.unsqueeze(-1) * relax_factor

        return q_seq

    def sample_temporal_sequence(self, full_data, batch_size, seq_length=300,
                                 required_node_ids=None, node_id_col=None):
        """
        Sample time subsequence for recursive q training.

        Uses NodeID-based alignment across frames to ensure the same spatial
        points are tracked consistently through time. Row-index-based alignment
        (iloc) was replaced because CSV row ordering may differ between frames,
        silently polluting the q recursion with mismatched node data.

        When required_node_ids is provided (fixed RF nodes from
        initialize_fixed_rf_points), the right-face batch uses EXACTLY those
        nodes every epoch, making the training RF estimate deterministic.

        Args:
            full_data: Full dataset DataFrame
            batch_size: Number of spatial points to sample
            seq_length: Number of consecutive time steps
            required_node_ids: Fixed node ID array for right face (stable RF).
                               When provided, right-face batch = these nodes.
            node_id_col: Column name for node IDs. Auto-detected if None.

        Returns:
            Dict with keys: coords_seq, u_data_seq, times, dt_seq, frames,
                            N_pts, N_t, right_indices, left_indices,
                            yz_face_indices, n_left, n_right, n_yz_faces, n_interior
        """
        # --- Auto-detect node ID column ---
        if node_id_col is None:
            for col in ['NodeLabel', 'Node', 'NID']:
                if col in full_data.columns:
                    node_id_col = col
                    break
        use_node_ids = (node_id_col is not None) and (node_id_col in full_data.columns)

        # --- Time sampling (stratified: dense in Prony identification zones) ---
        unique_times = np.sort(full_data['Time'].unique())
        N_times = len(unique_times)
        seq_length = min(seq_length, N_times)

        if seq_length >= N_times:
            time_indices = np.arange(N_times)
        else:
            # Four identification zones designed around the Prony time constants:
            #   (0,  12): ramp (τ=0.1s, 1s active). Frames 0-21 are exponential;
            #             uniform sampling only keeps t=0 and skips to t≈16 s,
            #             starving g0/g1 of gradient. Added to fix g0 underestimation.
            #   (10, 120): first-hold relaxation window (τ=10s vs τ=100s).
            #              At t=120s: τ=10s≈0, τ=100s≈33% → 7:1 contrast at t=32s.
            #   (300, 420): covers step-2 ramp transition + early hold (same logic).
            #   (600, 720): covers step-3 ramp transition + early hold.
            fast_zones = [(0, 12), (10, 120), (300, 420), (600, 720)]
            n_fast_per_zone = max(3, seq_length // 8)   # ~6 pts per zone for seq_length=50
            n_fast_total = n_fast_per_zone * len(fast_zones)
            n_slow = max(1, seq_length - n_fast_total)

            selected = set()
            selected.add(0)
            selected.add(N_times - 1)

            for t_lo, t_hi in fast_zones:
                zone_mask = (unique_times >= t_lo) & (unique_times <= t_hi)
                zone_idx  = np.where(zone_mask)[0]
                if len(zone_idx) > 0:
                    chosen = zone_idx[
                        np.round(np.linspace(0, len(zone_idx)-1, n_fast_per_zone)).astype(int)
                    ]
                    selected.update(chosen.tolist())

            uniform_all = np.round(np.linspace(0, N_times-1, n_slow)).astype(int)
            selected.update(uniform_all.tolist())

            time_indices = np.sort(np.array(list(selected), dtype=int))
            if len(time_indices) > seq_length:
                keep = np.round(np.linspace(0, len(time_indices)-1, seq_length)).astype(int)
                time_indices = time_indices[keep]

        sampled_times = unique_times[time_indices]
        assert np.all(np.diff(sampled_times) >= 0), "sampled_times not monotonic"

        # --- Spatial region identification from first frame ---
        first_slice = full_data[np.isclose(full_data['Time'].values, sampled_times[0], atol=1e-6)]

        x_coords = first_slice['X'].values
        y_coords = first_slice['Y'].values
        z_coords = first_slice['Z'].values
        x_min = self.bounds['x_min']; x_max = self.bounds['x_max']
        y_min = self.bounds['y_min']; y_max = self.bounds['y_max']
        z_min = self.bounds['z_min']; z_max = self.bounds['z_max']
        tol = 1e-4

        is_left_face  = np.abs(x_coords - x_min) < tol
        is_right_face = np.abs(x_coords - x_max) < tol
        is_y_min_face = np.abs(y_coords - y_min) < tol
        is_y_max_face = np.abs(y_coords - y_max) < tol
        is_z_min_face = np.abs(z_coords - z_min) < tol
        is_z_max_face = np.abs(z_coords - z_max) < tol
        is_yz_face    = is_y_min_face | is_y_max_face | is_z_min_face | is_z_max_face
        is_any_x_face = is_left_face | is_right_face
        is_interior   = ~is_any_x_face & ~is_yz_face

        left_idx     = np.where(is_left_face)[0]
        right_idx    = np.where(is_right_face)[0]
        yz_idx       = np.where(is_yz_face & ~is_any_x_face)[0]
        interior_idx = np.where(is_interior)[0]

        n_left     = max(5,  batch_size // 12)
        n_right    = max(10, batch_size // 7)
        n_yz_faces = max(15, batch_size // 7)
        n_interior = batch_size - n_left - n_right - n_yz_faces

        n_left     = min(n_left,     len(left_idx))
        n_right    = min(n_right,    len(right_idx))
        n_yz_faces = min(n_yz_faces, len(yz_idx))
        n_interior = min(n_interior, len(interior_idx))

        # --- Select node IDs for each region ---
        # Right face: use required_node_ids (fixed RF nodes) when provided
        rf_fixed_active = False  # tracks whether fixed nodes were actually used (for log)
        if required_node_ids is not None and use_node_ids and len(required_node_ids) > 0:
            first_right = first_slice.iloc[right_idx]
            mask_req    = first_right[node_id_col].isin(required_node_ids)
            req_found   = first_right[mask_req][node_id_col].values
            if len(req_found) == 0:
                # required nodes absent from first frame → fall back to random (rf_fixed stays False)
                sampled_right_rows = np.random.choice(right_idx, n_right, replace=False) if n_right > 0 else np.array([], dtype=int)
                right_node_ids = first_slice.iloc[sampled_right_rows][node_id_col].values
            else:
                rf_fixed_active = True
                if len(req_found) >= n_right:
                    # Enough required nodes: take first n_right (deterministic, same every epoch)
                    right_node_ids = req_found[:n_right]
                else:
                    # Fewer required nodes than n_right: supplement with random non-required
                    # right-face nodes to keep n_right — and therefore N_pts — constant.
                    # This is always feasible: n_right ≤ len(right_idx) = len(first_right),
                    # so other_right_ids has at least n_right - len(req_found) candidates.
                    other_right_ids = first_right[~mask_req][node_id_col].values
                    n_supplement    = min(n_right - len(req_found), len(other_right_ids))
                    if n_supplement > 0:
                        supplement_ids = np.random.choice(other_right_ids, n_supplement, replace=False)
                        right_node_ids = np.concatenate([req_found, supplement_ids])
                    else:
                        right_node_ids = req_found  # safety: shouldn't happen given constraints
                    n_right = len(right_node_ids)
        elif use_node_ids:
            sampled_right_rows = np.random.choice(right_idx, n_right, replace=False) if n_right > 0 else np.array([], dtype=int)
            right_node_ids = first_slice.iloc[sampled_right_rows][node_id_col].values
        else:
            sampled_right_rows = np.random.choice(right_idx, n_right, replace=False) if n_right > 0 else np.array([], dtype=int)
            right_node_ids = None

        # Left, YZ, interior: random sample each epoch, then get node IDs
        sampled_left_rows     = np.random.choice(left_idx,     n_left,     replace=False) if n_left     > 0 else np.array([], dtype=int)
        sampled_yz_rows       = np.random.choice(yz_idx,       n_yz_faces, replace=False) if n_yz_faces > 0 else np.array([], dtype=int)
        sampled_interior_rows = np.random.choice(interior_idx, n_interior, replace=False) if n_interior > 0 else np.array([], dtype=int)

        if use_node_ids:
            left_node_ids     = first_slice.iloc[sampled_left_rows][node_id_col].values     if n_left     > 0 else np.array([], dtype=int)
            yz_node_ids       = first_slice.iloc[sampled_yz_rows][node_id_col].values       if n_yz_faces > 0 else np.array([], dtype=int)
            interior_node_ids = first_slice.iloc[sampled_interior_rows][node_id_col].values if n_interior > 0 else np.array([], dtype=int)
            # Combined ordered node ID list: [right, left, yz, interior]
            all_node_ids = np.concatenate([right_node_ids, left_node_ids, yz_node_ids, interior_node_ids])
        else:
            # Fallback: combine row indices (may misalign if frames differ in row order)
            spatial_indices = np.concatenate([sampled_right_rows, sampled_left_rows, sampled_yz_rows, sampled_interior_rows])
            all_node_ids = None

        # Batch structure indices (position within the combined batch)
        required_indices_in_batch = np.arange(n_right)
        left_indices_in_batch     = np.arange(n_right, n_right + n_left)
        yz_face_indices_in_batch  = np.arange(n_right + n_left, n_right + n_left + n_yz_faces)

        self._left_indices_in_batch    = left_indices_in_batch
        self._yz_face_indices_in_batch = yz_face_indices_in_batch
        self._n_left     = n_left
        self._n_right    = n_right
        self._n_yz_faces = n_yz_faces
        self._n_interior = n_interior

        N_pts = n_right + n_left + n_yz_faces + n_interior

        if not hasattr(self, '_sampling_logged'):
            align_mode = f'NodeID({node_id_col})' if use_node_ids else 'RowIndex(no NodeID col—fallback)'
            rf_fixed   = 'yes' if rf_fixed_active else 'no'
            print(f"  [Sampling] Left={n_left}, Right={n_right}, YZ_faces={n_yz_faces}, Interior={n_interior}, Total={N_pts}")
            print(f"  [Alignment] {align_mode} | RF fixed={rf_fixed}")
            self._sampling_logged = True

        # --- Extract data per time step using NodeID alignment ---
        STRAIN_COLS = ['LE11', 'LE22', 'LE33', 'LE12', 'LE13', 'LE23']

        coords_seq     = []
        u_data_seq     = []
        strain_data_seq = []
        sampled_frames = []

        for t_val in sampled_times:
            time_slice = full_data[np.isclose(full_data['Time'].values, t_val, atol=1e-6)]
            frame_idx  = time_slice['Frame'].iloc[0] if len(time_slice) > 0 else None
            sampled_frames.append(frame_idx)

            if use_node_ids and all_node_ids is not None:
                # NodeID-based alignment: same nodes in same order across all frames
                ts_indexed = time_slice.set_index(node_id_col)
                batch_data = ts_indexed.reindex(all_node_ids).reset_index()
                # Missing nodes would corrupt q recursion — treat as hard error
                if batch_data[['X', 'Y', 'Z']].isnull().any().any():
                    n_missing = batch_data[['X', 'Y', 'Z']].isnull().any(axis=1).sum()
                    raise ValueError(
                        f"Frame {frame_idx}: {n_missing} sampled node(s) not found. "
                        f"CSV files may have inconsistent node sets across frames."
                    )
            else:
                # Fallback: row-index based (original behaviour, only when no NodeID col)
                batch_data = time_slice.iloc[spatial_indices]

            coords = torch.tensor(
                batch_data[['X', 'Y', 'Z']].values,
                dtype=torch.float32, device=device
            )
            u_data = torch.tensor(
                batch_data[['U1', 'U2', 'U3']].values,
                dtype=torch.float32, device=device
            )
            coords_seq.append(coords)
            u_data_seq.append(u_data)

            has_strain = all(c in batch_data.columns for c in STRAIN_COLS)
            if has_strain:
                strain_fe = batch_data[STRAIN_COLS].values.astype(np.float64)
                strain_data_seq.append(torch.tensor(strain_fe, dtype=torch.float32, device=device))
            else:
                strain_data_seq.append(None)

        dt_seq = torch.tensor(np.diff(sampled_times), dtype=torch.float32, device=device)

        result = {
            'coords_seq':      coords_seq,
            'u_data_seq':      u_data_seq,
            'strain_data_seq': strain_data_seq,
            'times':           sampled_times,
            'frames':          sampled_frames,
            'dt_seq':          dt_seq,
            'N_pts':           N_pts,
            'N_t':             len(sampled_times),
            'right_indices':   required_indices_in_batch,
            'left_indices':    left_indices_in_batch,
            'yz_face_indices': yz_face_indices_in_batch,
            'n_left':          n_left,
            'n_right':         n_right,
            'n_yz_faces':      n_yz_faces,
            'n_interior':      n_interior,
        }

        return result

    def __init__(self, model, mat_params, fe_loader, bounds,
                 lambda_data=100.0, lambda_pde=0.3, lambda_bc_left=100.0, lambda_bc_right=10000.0,
                 lambda_traction=15.0, lambda_rf=5.0, lambda_strain=10.0):
        """
        Initialize inverse PINN solver with constant loss weights.
        """
        self.model = model
        self.mat = mat_params
        self.fe_loader = fe_loader
        self.bounds = bounds

        self.lambda_data = lambda_data
        self.lambda_pde = lambda_pde
        self.lambda_bc_left = lambda_bc_left
        self.lambda_bc_right = lambda_bc_right
        self.lambda_traction = lambda_traction
        self.lambda_rf = lambda_rf
        self.lambda_strain = lambda_strain

        # Reference scales for normalization
        self.x_min = bounds['x_min']; self.x_max = bounds['x_max']
        self.y_min = bounds['y_min']; self.y_max = bounds['y_max']
        self.z_min = bounds['z_min']; self.z_max = bounds['z_max']
        self.t_min = bounds['t_min']; self.t_max = bounds['t_max']
        self.Lx = self.x_max - self.x_min
        self.Ly = self.y_max - self.y_min
        self.Lz = self.z_max - self.z_min
        self.T_max = max(self.t_max - self.t_min, 1.0)
        self.L_ref = self.Lx  # default length scale (mm)
        self.E_ref = 1.0      # Stress reference (MPa)

        # Loss history
        self.loss_history = []
        self.loss_components_history = []
        self.param_history = []
        self.weight_history = []  # Track weight evolution (for plotting)

        # Fixed RF sampling points (initialized in train())
        self.rf_node_ids = None
        self.rf_node_col = None

        print(f"  λ_data={lambda_data}, λ_pde={lambda_pde}, λ_strain={lambda_strain}")
        print(f"  λ_bc_left={lambda_bc_left}, λ_bc_right={lambda_bc_right}")
        print(f"  λ_rf={lambda_rf}, λ_traction={lambda_traction}")

    def initialize_fixed_rf_points(self, n_rf_points=512):
        """
        Initialize fixed set of right face points for stable RF computation.

        CRITICAL: Use same spatial points across all time steps to convert RF from
        high-variance Monte Carlo estimate to stable deterministic estimate.

        Args:
            n_rf_points: Number of fixed points to sample from right face
        """
        # Get right boundary node IDs from FEDataLoader
        node_ids, node_col = self.fe_loader.get_boundary_node_ids(face='right', tol=1e-4)

        # Sample fixed subset (or use all if fewer than n_rf_points)
        if len(node_ids) > n_rf_points:
            np.random.seed(42)  # Reproducible sampling
            self.rf_node_ids = np.random.choice(node_ids, n_rf_points, replace=False)
        else:
            self.rf_node_ids = node_ids

        self.rf_node_col = node_col

        print(f"\n  Fixed RF sampling: {len(self.rf_node_ids)} nodes from right face")
        print(f"  (Converts RF from Monte Carlo to stable deterministic estimate)")

    def compute_strain(self, u, x, y, z):
        """
        【CRITICAL FIX】Compute strain with ENGINEERING SHEAR definition.

        Engineering Voigt notation: [ε11, ε22, ε33, γ12, γ13, γ23]
        where γ_ij = ∂u_i/∂x_j + ∂u_j/∂x_i (NOT divided by 2!)

        This is required to match data convention and stiffness matrix multiplication.
        Rotation will handle the tensor/Voigt conversion internally.

        Returns: strain in ENGINEERING Voigt notation (N×6)
        """
        u1, u2, u3 = u[:, 0:1], u[:, 1:2], u[:, 2:3]

        # Compute gradients
        du1_dx = torch.autograd.grad(u1, x, torch.ones_like(u1), create_graph=True)[0]
        du1_dy = torch.autograd.grad(u1, y, torch.ones_like(u1), create_graph=True)[0]
        du1_dz = torch.autograd.grad(u1, z, torch.ones_like(u1), create_graph=True)[0]

        du2_dx = torch.autograd.grad(u2, x, torch.ones_like(u2), create_graph=True)[0]
        du2_dy = torch.autograd.grad(u2, y, torch.ones_like(u2), create_graph=True)[0]
        du2_dz = torch.autograd.grad(u2, z, torch.ones_like(u2), create_graph=True)[0]

        du3_dx = torch.autograd.grad(u3, x, torch.ones_like(u3), create_graph=True)[0]
        du3_dy = torch.autograd.grad(u3, y, torch.ones_like(u3), create_graph=True)[0]
        du3_dz = torch.autograd.grad(u3, z, torch.ones_like(u3), create_graph=True)[0]

        # Normal strains
        e11 = du1_dx
        e22 = du2_dy
        e33 = du3_dz

        # ENGINEERING shear strains (γ = ∂ui/∂xj + ∂uj/∂xi, NO factor of 0.5!)
        gamma12 = du1_dy + du2_dx
        gamma13 = du1_dz + du3_dx
        gamma23 = du2_dz + du3_dy

        strain = torch.cat([e11, e22, e33, gamma12, gamma13, gamma23], dim=1)

        return strain

    def to_material_coords(self, voigt_global_eng):
        """
        【A3-CRITICAL FIX】Transform engineering Voigt from GLOBAL → MATERIAL coordinates.

        Rotation direction: Q^T @ T @ Q
        (where Q is the material→global rotation matrix, so Q^T is global→material)

        Proper procedure for engineering Voigt [ε11, ε22, ε33, γ12, γ13, γ23]:
        1. Convert engineering Voigt → tensor (shear / 2)
        2. Rotate tensor: T_material = Q^T @ T_global @ Q
        3. Convert tensor → engineering Voigt (shear * 2)

        Args:
            voigt_global_eng: (N×6) in GLOBAL coordinates, ENGINEERING Voigt

        Returns:
            voigt_material_eng: (N×6) in MATERIAL coordinates, ENGINEERING Voigt
        """
        N = voigt_global_eng.shape[0]

        # STEP 1: Engineering Voigt → Tensor (shear / 2)
        tensor = torch.zeros(N, 3, 3, device=device)
        tensor[:, 0, 0] = voigt_global_eng[:, 0]  # ε11
        tensor[:, 1, 1] = voigt_global_eng[:, 1]  # ε22
        tensor[:, 2, 2] = voigt_global_eng[:, 2]  # ε33
        tensor[:, 0, 1] = voigt_global_eng[:, 3] / 2.0  # γ12 → ε12
        tensor[:, 1, 0] = voigt_global_eng[:, 3] / 2.0
        tensor[:, 0, 2] = voigt_global_eng[:, 4] / 2.0  # γ13 → ε13
        tensor[:, 2, 0] = voigt_global_eng[:, 4] / 2.0
        tensor[:, 1, 2] = voigt_global_eng[:, 5] / 2.0  # γ23 → ε23
        tensor[:, 2, 1] = voigt_global_eng[:, 5] / 2.0

        # STEP 2: Rotate GLOBAL → MATERIAL: T_mat = Q^T @ T_glob @ Q
        Q = self.mat.Q
        rotated = torch.einsum('ji,njk,kl->nil', Q, tensor, Q)  # Q^T @ T @ Q

        # STEP 3: Tensor → Engineering Voigt (shear * 2)
        voigt_material_eng = torch.zeros(N, 6, device=device)
        voigt_material_eng[:, 0] = rotated[:, 0, 0]
        voigt_material_eng[:, 1] = rotated[:, 1, 1]
        voigt_material_eng[:, 2] = rotated[:, 2, 2]
        voigt_material_eng[:, 3] = 2.0 * rotated[:, 0, 1]
        voigt_material_eng[:, 4] = 2.0 * rotated[:, 0, 2]
        voigt_material_eng[:, 5] = 2.0 * rotated[:, 1, 2]

        return voigt_material_eng

    def to_global_coords(self, voigt_material_eng):
        """
        【A3-CRITICAL FIX】Transform engineering Voigt from MATERIAL → GLOBAL coordinates.

        Rotation direction: Q @ T @ Q^T
        (where Q is the material→global rotation matrix)

        Proper procedure for engineering Voigt [ε11, ε22, ε33, γ12, γ13, γ23]:
        1. Convert engineering Voigt → tensor (shear / 2)
        2. Rotate tensor: T_global = Q @ T_material @ Q^T
        3. Convert tensor → engineering Voigt (shear * 2)

        Args:
            voigt_material_eng: (N×6) in MATERIAL coordinates, ENGINEERING Voigt

        Returns:
            voigt_global_eng: (N×6) in GLOBAL coordinates, ENGINEERING Voigt
        """
        N = voigt_material_eng.shape[0]

        # STEP 1: Engineering Voigt → Tensor (shear / 2)
        tensor = torch.zeros(N, 3, 3, device=device)
        tensor[:, 0, 0] = voigt_material_eng[:, 0]
        tensor[:, 1, 1] = voigt_material_eng[:, 1]
        tensor[:, 2, 2] = voigt_material_eng[:, 2]
        tensor[:, 0, 1] = voigt_material_eng[:, 3] / 2.0
        tensor[:, 1, 0] = voigt_material_eng[:, 3] / 2.0
        tensor[:, 0, 2] = voigt_material_eng[:, 4] / 2.0
        tensor[:, 2, 0] = voigt_material_eng[:, 4] / 2.0
        tensor[:, 1, 2] = voigt_material_eng[:, 5] / 2.0
        tensor[:, 2, 1] = voigt_material_eng[:, 5] / 2.0

        # STEP 2: Rotate MATERIAL → GLOBAL: T_glob = Q @ T_mat @ Q^T
        Q = self.mat.Q
        rotated = torch.einsum('ij,njk,lk->nil', Q, tensor, Q)  # Q @ T @ Q^T

        # STEP 3: Tensor → Engineering Voigt (shear * 2)
        voigt_global_eng = torch.zeros(N, 6, device=device)
        voigt_global_eng[:, 0] = rotated[:, 0, 0]
        voigt_global_eng[:, 1] = rotated[:, 1, 1]
        voigt_global_eng[:, 2] = rotated[:, 2, 2]
        voigt_global_eng[:, 3] = 2.0 * rotated[:, 0, 1]
        voigt_global_eng[:, 4] = 2.0 * rotated[:, 0, 2]
        voigt_global_eng[:, 5] = 2.0 * rotated[:, 1, 2]

        return voigt_global_eng

    def rotate_stress_to_global(self, stress_material):
        """
        Rotate STRESS (not strain!) from MATERIAL → GLOBAL coordinates.

        CRITICAL: Stress uses Voigt notation [σ11, σ22, σ33, σ12, σ13, σ23]
        where σ_ij are tensor components directly - NO /2 or *2 conversions!

        This differs from strain rotation which needs γ/2 and *2 conversions.

        Args:
            stress_material: (N×6) stress in MATERIAL coordinates
                            [σ11_mat, σ22_mat, σ33_mat, σ12_mat, σ13_mat, σ23_mat]

        Returns:
            stress_global: (N×6) stress in GLOBAL coordinates
                          [σ11_glob, σ22_glob, σ33_glob, σ12_glob, σ13_glob, σ23_glob]
        """
        N = stress_material.shape[0]

        # STEP 1: Voigt → Tensor (NO division by 2 for stress!)
        tensor = torch.zeros(N, 3, 3, device=device)
        tensor[:, 0, 0] = stress_material[:, 0]  # σ11
        tensor[:, 1, 1] = stress_material[:, 1]  # σ22
        tensor[:, 2, 2] = stress_material[:, 2]  # σ33
        tensor[:, 0, 1] = stress_material[:, 3]  # σ12 (NOT /2!)
        tensor[:, 1, 0] = stress_material[:, 3]
        tensor[:, 0, 2] = stress_material[:, 4]  # σ13 (NOT /2!)
        tensor[:, 2, 0] = stress_material[:, 4]
        tensor[:, 1, 2] = stress_material[:, 5]  # σ23 (NOT /2!)
        tensor[:, 2, 1] = stress_material[:, 5]

        # STEP 2: Rotate MATERIAL → GLOBAL: σ_glob = Q @ σ_mat @ Q^T
        Q = self.mat.Q
        rotated = torch.einsum('ij,njk,lk->nil', Q, tensor, Q)

        # STEP 3: Tensor → Voigt (NO multiplication by 2 for stress!)
        stress_global = torch.zeros(N, 6, device=device)
        stress_global[:, 0] = rotated[:, 0, 0]
        stress_global[:, 1] = rotated[:, 1, 1]
        stress_global[:, 2] = rotated[:, 2, 2]
        stress_global[:, 3] = rotated[:, 0, 1]  # (NOT *2!)
        stress_global[:, 4] = rotated[:, 0, 2]  # (NOT *2!)
        stress_global[:, 5] = rotated[:, 1, 2]  # (NOT *2!)

        return stress_global

    def compute_stress(self, strain_global, q):
        """
        Component-wise Prony relaxation aligned with EX1/EX3.

        Form used in this code:
            σ = C^∞ : ε + Σ_k C^(k) : (ε - q^(k))

        Notes on q:
        - q is the branch internal strain variable from compute_q_recursive.
        - During a hold, q approaches ε, so each branch contribution relaxes out.

        Args:
            strain_global: (N×6) strain in GLOBAL coordinates
            q: (N×6×N_prony) internal variables for all branches

        Returns:
            stress_global: (N×6) stress in GLOBAL coordinates
        """
        # 1. Rotate strain to MATERIAL coordinates (45° off-axis)
        strain_material = self.to_material_coords(strain_global)

        N = strain_material.shape[0]

        # 2. Get long-term stiffness C^∞
        C_inf = self.mat.get_stiffness_matrix_inf()  # (6×6)

        # 3. Compute equilibrium stress: σ_eq = C^∞ : ε
        stress_eq = torch.einsum('ij,nj->ni', C_inf, strain_material)  # (N×6)

        # 4. Compute branch relaxation stress: Σ_k C^(k) : (ε - q^(k))
        stress_relax = torch.zeros(N, 6, device=device, dtype=torch.float32)

        for k in range(self.mat.N_prony):
            C_k = self.mat.get_stiffness_matrix_branch(k)  # (6×6)
            q_k = q[:, :, k]  # (N×6)
            stress_relax += torch.einsum('ij,nj->ni', C_k, strain_material - q_k)  # (N×6)

        # 5. Total stress: σ = C^∞ : ε + Σ_k C^(k) : (ε - q^(k))
        stress_material = stress_eq + stress_relax

        # 6. Rotate stress back to GLOBAL coordinates
        stress_global = self.rotate_stress_to_global(stress_material)

        return stress_global

    def compute_pde_residual(self, stress, x, y, z):
        """
        Compute PDE residual: ∇·σ = 0 (equilibrium equation)

        Returns: residual vector (N×3)
        """
        s11, s22, s33 = stress[:, 0:1], stress[:, 1:2], stress[:, 2:3]
        s12, s13, s23 = stress[:, 3:4], stress[:, 4:5], stress[:, 5:6]

        # Divergence components
        div_x = (torch.autograd.grad(s11, x, torch.ones_like(s11), create_graph=True)[0] +
                 torch.autograd.grad(s12, y, torch.ones_like(s12), create_graph=True)[0] +
                 torch.autograd.grad(s13, z, torch.ones_like(s13), create_graph=True)[0])

        div_y = (torch.autograd.grad(s12, x, torch.ones_like(s12), create_graph=True)[0] +
                 torch.autograd.grad(s22, y, torch.ones_like(s22), create_graph=True)[0] +
                 torch.autograd.grad(s23, z, torch.ones_like(s23), create_graph=True)[0])

        div_z = (torch.autograd.grad(s13, x, torch.ones_like(s13), create_graph=True)[0] +
                 torch.autograd.grad(s23, y, torch.ones_like(s23), create_graph=True)[0] +
                 torch.autograd.grad(s33, z, torch.ones_like(s33), create_graph=True)[0])

        residual = torch.cat([div_x, div_y, div_z], dim=1)

        return residual

    def compute_reaction_force(self, stress_bc_global):
        """
        Compute reaction force from stress on right boundary.

        Args:
            stress_bc_global: (N_bc, 6) stress tensor in global coords

        Returns:
            rf_pred: scalar, reaction force in N
        """
        sigma_11 = stress_bc_global[:, 0]
        y_range = self.bounds['y_max'] - self.bounds['y_min']
        z_range = self.bounds['z_max'] - self.bounds['z_min']
        area = y_range * z_range
        rf_pred = sigma_11.mean() * area
        return rf_pred

    def get_rf_target_by_frame(self, frame_idx, rf_data):
        """
        Get RF target value by Frame index (STRICT ALIGNMENT).

        CRITICAL: Use Frame as primary key to ensure RF, U, and field data
        are from the exact same Abaqus output step, avoiding float Time mismatch.

        Args:
            frame_idx: Integer frame number
            rf_data: DataFrame with 'Frame' and 'RF' columns

        Returns:
            RF value (scalar tensor)
        """
        # Find RF entry matching this frame
        rf_row = rf_data[rf_data['Frame'] == frame_idx]

        if len(rf_row) == 0:
            # Frame not found in RF data - use closest available frame
            closest_frame = rf_data.iloc[(rf_data['Frame'] - frame_idx).abs().argsort()[:1]]
            rf_value = closest_frame['RF'].values[0]
            print(f"Warning: Frame {frame_idx} not in RF data, using closest Frame {closest_frame['Frame'].values[0]}")
        else:
            rf_value = rf_row['RF'].values[0]

        return torch.tensor(rf_value, dtype=torch.float32, device=device)


    def train(self, epochs=10000, batch_size=1024,
              lr_network=2e-3, lr_elastic=5e-3, lr_params=1e-2,
              log_interval=100):
        """
        Train the inverse PINN with constant loss weights and memory-optimized
        time sequence sampling for q recursion.

        MEMORY OPTIMIZATION STRATEGY:
        - Spatial batch: 128 points (down from 512) to save memory
        - Time window: 50 steps (NOT full 900!) to prevent OOM
        - t=0 anchoring via stratified sequence sampling
        - Gradient truncation: Detach every 50 steps in q recursion

        RATIONALE:
        - Full 900-step backprop causes OOM (43 GiB > 16 GiB GPU)
        - 50-step windows balance memory and gradient flow
        - Frequent t=0 starts ensure q(t=0)=0 consistency

        Args:
            epochs: Number of training epochs
            batch_size: Spatial points per batch (recommend 64-128)
            lr_network: Learning rate for displacement network
            lr_elastic: Learning rate for elastic constants (C_ij)
            lr_params: Learning rate for Prony spectrum weights
            log_interval: Print frequency
        """

        start_time = time.time()

        full_data = self.fe_loader.full_data

        rf_times = torch.tensor(self.fe_loader.rf_data['Time'].values, dtype=torch.float32, device=device)
        rf_values = torch.tensor(self.fe_loader.rf_data['RF'].values, dtype=torch.float32, device=device)

        # Per-point relative normalization floor.
        # floor=20N gave tail/peak weight ratio of 115x, causing g3(τ=100s) to
        # dominate (observed g3≈0.77 vs target 0.20) while short-τ branches
        # received essentially no gradient signal.
        # floor=100N reduces the ratio to ~20x: the relaxation tail still gets
        # more emphasis than an absolute scale, but the fast dynamics (τ<10s)
        # are no longer suppressed by 2 orders of magnitude.
        self.rf_rel_floor = torch.tensor(100.0, dtype=torch.float32, device=device)

        # Compute displacement reference scale from right boundary data
        # Extract right face displacement magnitude for normalization
        x_max = self.bounds['x_max']
        right_face_data = full_data[np.abs(full_data['X'].values - x_max) < 1e-4]
        u_right = right_face_data[['U1', 'U2', 'U3']].values
        u_magnitude = np.linalg.norm(u_right, axis=1)
        u_ref = np.max(u_magnitude)  # Maximum displacement magnitude
        self.u_ref_sq = (u_ref + 1e-10) ** 2  # Add small epsilon to avoid division by zero

        print(f"RF relative floor: {self.rf_rel_floor.item():.1f} N (per-point relative normalization)")
        print(f"RF range: [{rf_values.min().item():.2f}, {rf_values.max().item():.2f}] N")
        print(f"U scale (max|u|): {u_ref:.6f} mm")
        print(f"Normalization: RF by mean, U by max")

        # Initialize fixed RF sampling points for stable RF computation
        self.initialize_fixed_rf_points(n_rf_points=512)

        for param in self.model.parameters():
            param.requires_grad = True
        for param in self.mat.parameters():
            param.requires_grad = True

        # Scheme B training with constant weights:
        #   - Network LR: learns u(x,y,z,t) from data+strain
        #   - Elastic LR: learns C11, C22, C66
        #   - Spectrum LR: learns the single amplitude parameter A_g
        elastic_params = [
            self.mat.logC11_total,
            self.mat.logC22_total,
            self.mat.logC66_total
        ]
        optimizer = Adam([
            {'params': self.model.parameters(),   'lr': lr_network},
            {'params': elastic_params,            'lr': lr_elastic},
            {'params': [self.mat.logit_Ag],       'lr': lr_params}
        ])
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=epochs, eta_min=1e-5
        )

        for epoch in range(epochs):
            self.model.train()

            optimizer.zero_grad()

            def norm_coords(x, y, z, t):
                x_hat = (x - self.x_min) / self.Lx
                y_hat = (y - self.y_min) / self.Ly
                z_hat = (z - self.z_min) / self.Lz
                t_hat = (t - self.t_min) / self.T_max
                return x_hat, y_hat, z_hat, t_hat

            # Time subsampling: stratified with dense coverage of 10-120 s identification
            # zones (one per loading hold) where τ=10 s and τ=100 s are distinguishable.
            # Each zone gets ~6 pts (n_fast_per_zone = seq_length//8); remaining fills
            # uniformly.  Uniform-only (~18 s spacing) was the root cause of g2/g3 swap.
            seq_length_efficient = 50

            seq_data = self.sample_temporal_sequence(
                full_data,
                batch_size=batch_size,
                seq_length=seq_length_efficient,  # Subsampled for efficiency
                required_node_ids=self.rf_node_ids,
                node_id_col=self.rf_node_col
            )

            coords_seq = seq_data['coords_seq']
            u_data_seq = seq_data['u_data_seq']
            strain_data_seq = seq_data.get('strain_data_seq', [None] * len(seq_data['times']))
            sampled_times = seq_data['times']
            sampled_frames = seq_data['frames']  # CRITICAL: Frame indices for strict RF/U alignment
            dt_seq = seq_data['dt_seq']
            N_pts = seq_data['N_pts']
            N_t = seq_data['N_t']
            # Stratified sampling indices
            rf_indices = seq_data['right_indices']    # Right face nodes for RF
            left_indices = seq_data['left_indices']   # Left face nodes for BC
            yz_face_indices = seq_data['yz_face_indices']  # FIX #4: Y/Z face nodes for traction BC
            n_interior = seq_data['n_interior']       # Interior points for PDE
            n_yz_faces = seq_data['n_yz_faces']       # FIX #4

            # Verify sampling worked
            if len(rf_indices) == 0:
                raise RuntimeError("No RF nodes in batch!")
            if len(left_indices) == 0:
                print("  WARNING: No left boundary nodes in batch - BC_left loss will be 0")
            if len(yz_face_indices) == 0:
                print("  WARNING: No Y/Z face nodes in batch - traction loss will be 0 (affects C23!)")

            strain_seq = []
            u_pred_seq = []
            coord_grads_seq = []  # (x_c, y_c, z_c) per time step for PDE spatial derivatives

            for n in range(N_t):
                coords = coords_seq[n]
                t_val = sampled_times[n]
                t_tensor = torch.full((N_pts, 1), t_val, dtype=torch.float32, device=device)

                # Single forward pass with gradient-enabled coordinates.
                # The previous code ran two separate forwards (one without grad for u_pred,
                # one with grad for strain), producing inconsistent computation graphs.
                # Now u_pred and strain share the same graph, and coord_grads_seq stores
                # the leaf coordinate tensors for later use in the PDE residual.
                x_c = coords[:, 0:1].clone().requires_grad_(True)
                y_c = coords[:, 1:2].clone().requires_grad_(True)
                z_c = coords[:, 2:3].clone().requires_grad_(True)
                x_hat_c, y_hat_c, z_hat_c, t_hat_c = norm_coords(x_c, y_c, z_c, t_tensor)
                u_pred = self.model(x_hat_c, y_hat_c, z_hat_c, t_hat_c)
                u_pred_seq.append(u_pred)
                strain = self.compute_strain(u_pred, x_c, y_c, z_c)
                strain_seq.append(strain)
                coord_grads_seq.append((x_c, y_c, z_c))

            strain_seq_tensor = torch.stack(strain_seq, dim=0)

            # 【A3 FIX】Rotate strain to MATERIAL coordinates before q recursion
            # q MUST evolve in the same coordinate system as the constitutive law
            # Otherwise 45° rotation causes systematic coupling errors
            strain_mat_seq = []
            for n in range(N_t):
                strain_global_n = strain_seq_tensor[n]  # (N_pts, 6)
                strain_mat_n = self.to_material_coords(strain_global_n)
                strain_mat_seq.append(strain_mat_n)
            strain_mat_seq_tensor = torch.stack(strain_mat_seq, dim=0)  # (N_t, N_pts, 6)

            # ========== 【A2】Compute q via Discrete Recursion ==========
            # NO network output for q! Computed from MATERIAL-frame strain history
            # CRITICAL FIX: Keep gradient flow via truncated backprop (q_prev detached in recursion)
            q_seq = self.compute_q_recursive(strain_mat_seq_tensor, dt_seq, self.mat.rho)

            # ========== PERFORMANCE: Extract RF data from main batch (no duplicate forward!) ==========
            # RF nodes are already in main batch (via required_node_ids)
            # Just extract using indices - eliminates ~50% computation overhead!
            rf_indices_torch = torch.tensor(rf_indices, dtype=torch.long, device=device)

            # ========== Compute Losses Over Time Sequence ==========
            loss_data_total = torch.tensor(0.0, device=device)
            loss_strain_total = torch.tensor(0.0, device=device)
            loss_rf_total = torch.tensor(0.0, device=device)
            loss_bc_left_total = torch.tensor(0.0, device=device)
            loss_bc_right_total = torch.tensor(0.0, device=device)
            loss_traction_total = torch.tensor(0.0, device=device)
            loss_pde_total = torch.tensor(0.0, device=device)

            for n in range(N_t):
                coords = coords_seq[n]
                u_data = u_data_seq[n]
                u_pred = u_pred_seq[n]
                strain_n = strain_seq[n]
                q_n = q_seq[n]
                t_val = sampled_times[n]

                # Data loss: Normalized MSE between predicted and target displacement
                # Normalize by U_ref^2 to make it dimensionless and comparable with BC losses
                loss_data_total += torch.mean((u_pred - u_data) ** 2) / self.u_ref_sq

                # Strain loss: directly constrain AD strain to match FE LE strain
                # This pins ε(x,y,z,t) and eliminates (u, g_k) joint ill-posedness.
                # LE ≈ linearized strain (valid for small-to-moderate deformation).
                if strain_data_seq[n] is not None:
                    strain_fe_n = strain_data_seq[n]  # (N_pts, 6) from CSV LE columns
                    eps_ref_sq = 0.30 ** 2             # normalize by max applied strain
                    loss_strain_total += torch.mean((strain_n - strain_fe_n) ** 2) / eps_ref_sq

                # Compute stress from strain and q (for data / BC / RF)
                stress_n = self.compute_stress(strain_n, q_n)

                # PDE loss: compute every 30 time steps (~2 evaluations per 50-step epoch)
                pde_sample_interval = 30
                compute_pde = (n % pde_sample_interval == 0) and (n_interior > 0)

                if compute_pde:
                    # Compute PDE residual using the full-batch leaf tensors from
                    # coord_grads_seq[n].  stress_pde depends on those leaves through
                    # strain_seq[n] (computed with create_graph=True), so we differentiate
                    # w.r.t. the full leaf and slice the resulting gradient to the interior
                    # block.  This avoids the "tensor not in graph" error that arises when
                    # differentiating w.r.t. a slice of a leaf (slice is not a graph node).
                    interior_start = len(rf_indices) + len(left_indices) + len(yz_face_indices)
                    x_c_n, y_c_n, z_c_n = coord_grads_seq[n]   # full-batch leaf tensors

                    strain_pde = strain_seq[n][interior_start:]
                    q_pde      = q_n[interior_start:, :, :]
                    stress_pde = self.compute_stress(strain_pde, q_pde)

                    s11 = stress_pde[:, 0:1]; s22 = stress_pde[:, 1:2]; s33 = stress_pde[:, 2:3]
                    s12 = stress_pde[:, 3:4]; s13 = stress_pde[:, 4:5]; s23 = stress_pde[:, 5:6]

                    def _pde_grad(s, leaf):
                        """∂s/∂leaf for interior points via full-leaf differentiation."""
                        g = torch.autograd.grad(
                            s, leaf, torch.ones_like(s),
                            create_graph=True, allow_unused=True
                        )[0]
                        return g[interior_start:] if g is not None else torch.zeros_like(s)

                    div_x = _pde_grad(s11, x_c_n) + _pde_grad(s12, y_c_n) + _pde_grad(s13, z_c_n)
                    div_y = _pde_grad(s12, x_c_n) + _pde_grad(s22, y_c_n) + _pde_grad(s23, z_c_n)
                    div_z = _pde_grad(s13, x_c_n) + _pde_grad(s23, y_c_n) + _pde_grad(s33, z_c_n)
                    pde_res = torch.cat([div_x, div_y, div_z], dim=1)

                    # Dimensional normalization: div σ has dimension [MPa/mm]
                    pde_res_norm_sq = torch.sum(pde_res ** 2, dim=1)
                    sigma_ref = 100.0  # MPa
                    L_ref = self.Lx   # mm
                    pde_ref_sq = (sigma_ref / L_ref) ** 2
                    relative_pde_res = pde_res_norm_sq / pde_ref_sq
                    loss_pde_total += torch.mean(relative_pde_res)

                # RF loss: keep strain/q connected to the displacement graph so RF
                # constrains the inferred strain history together with the network.
                rf_strain_n = strain_seq[n][rf_indices_torch, :]
                rf_q_n = q_seq[n][rf_indices_torch, :, :]

                # Compute stress at RF nodes
                stress_rf = self.compute_stress(rf_strain_n, rf_q_n)
                rf_pred = self.compute_reaction_force(stress_rf)

                # Frame-based RF target lookup
                frame_idx = sampled_frames[n]
                rf_target = self.get_rf_target_by_frame(frame_idx, self.fe_loader.rf_data)

                # Per-point relative MSE: error weighted by 1/|RF_target|.
                # Gives equal relative importance to relaxation tail (small RF)
                # and loading peak (large RF), making the loss sensitive to the
                # full relaxation curve shape—critical for Prony identification.
                rf_denom = torch.clamp(torch.abs(rf_target), min=self.rf_rel_floor)
                rf_error_rel = (rf_pred - rf_target) / rf_denom
                loss_rf_total += rf_error_rel ** 2

                # BC right loss: Extract from main batch (same RF nodes)
                u_bc_right_pred = u_pred_seq[n][rf_indices_torch, :]  # From main batch
                u_bc_right_target = u_data_seq[n][rf_indices_torch, :]  # From main batch
                loss_bc_right_total += torch.mean((u_bc_right_pred - u_bc_right_target) ** 2) / self.u_ref_sq

                # BC left loss: Use pre-sampled LEFT FACE points (guaranteed to be in batch)
                # Left boundary is fixed (u=0), Dirichlet BC
                if len(left_indices) > 0:
                    left_indices_torch = torch.tensor(left_indices, dtype=torch.long, device=device)
                    u_bc_left_pred = u_pred[left_indices_torch, :]
                    # Target is zero displacement
                    loss_bc_left_total += torch.mean(u_bc_left_pred ** 2) / self.u_ref_sq

                # Traction-free BC: Lateral surfaces (y and z faces)
                # CRITICAL: These constraints enhance sensitivity to C22, C23, C44, C66
                # FIX #4: Use pre-sampled Y/Z face indices (guaranteed to be in batch!)
                y_min, y_max = self.bounds['y_min'], self.bounds['y_max']
                z_min, z_max = self.bounds['z_min'], self.bounds['z_max']
                tol = 1e-4

                # FIX #4: Use guaranteed Y/Z face points from stratified sampling
                if len(yz_face_indices) > 0:
                    yz_indices_torch = torch.tensor(yz_face_indices, dtype=torch.long, device=device)
                    coords_yz = coords[yz_indices_torch, :]
                    stress_yz = stress_n[yz_indices_torch, :]

                    # Distinguish Y vs Z faces within the pre-sampled points
                    # Y-faces: σ·n = [σ_xy, σ_yy, σ_yz] = 0 (n = [0, ±1, 0])
                    is_y_face = (torch.abs(coords_yz[:, 1] - y_min) < tol) | (torch.abs(coords_yz[:, 1] - y_max) < tol)
                    # Z-faces: σ·n = [σ_xz, σ_yz, σ_zz] = 0 (n = [0, 0, ±1])
                    is_z_face = (torch.abs(coords_yz[:, 2] - z_min) < tol) | (torch.abs(coords_yz[:, 2] - z_max) < tol)

                    sigma_ref_sq = 100.0 ** 2  # MPa²: normalize to make traction loss dimensionless
                    if is_y_face.any():
                        stress_y = stress_yz[is_y_face, :]
                        # Traction on Y-face: t_y = [σ_xy, σ_yy, σ_yz] should be 0
                        traction_y = torch.stack([stress_y[:, 3], stress_y[:, 1], stress_y[:, 5]], dim=1)
                        loss_traction_total += torch.mean(traction_y ** 2) / sigma_ref_sq

                    if is_z_face.any():
                        stress_z = stress_yz[is_z_face, :]
                        # Traction on Z-face: t_z = [σ_xz, σ_yz, σ_zz] should be 0
                        traction_z = torch.stack([stress_z[:, 4], stress_z[:, 5], stress_z[:, 2]], dim=1)
                        loss_traction_total += torch.mean(traction_z ** 2) / sigma_ref_sq

            # Average losses over time sequence
            loss_data     = loss_data_total     / N_t
            loss_strain   = loss_strain_total   / N_t
            loss_rf       = loss_rf_total       / N_t
            loss_bc_left  = loss_bc_left_total  / N_t
            loss_bc_right = loss_bc_right_total / N_t
            loss_traction = loss_traction_total / N_t
            # PDE is computed every pde_sample_interval steps; divide by the actual
            # number of steps evaluated, not N_t, to avoid ~15x weight dilution.
            n_pde_computed = max(1, sum(1 for i in range(N_t) if i % pde_sample_interval == 0))
            loss_pde = loss_pde_total / n_pde_computed

            # CRITICAL: Add parameter penalty to prevent shrinking stiffness
            loss_param_penalty = self.mat.compute_parameter_penalty()

            # CRITICAL FIX: Increase penalty weight to strongly enforce physical bounds
            loss_total = (self.lambda_data     * loss_data +
                         self.lambda_strain    * loss_strain +
                         self.lambda_bc_left   * loss_bc_left +
                         self.lambda_bc_right  * loss_bc_right +
                         self.lambda_traction  * loss_traction +
                         self.lambda_rf        * loss_rf +
                         self.lambda_pde       * loss_pde +
                         50.0                  * loss_param_penalty)

            loss_total.backward()

            # Clip each param group independently
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            torch.nn.utils.clip_grad_norm_(elastic_params, max_norm=0.5)
            torch.nn.utils.clip_grad_norm_([self.mat.logit_Ag], max_norm=0.5)

            # g0 and g1 are fixed; logit_Ag controls the total matrix relaxation amplitude.
            optimizer.step()
            scheduler.step()

            # Store history with detailed components
            self.loss_history.append(loss_total.item())
            self.loss_components_history.append({
                'data': loss_data.item(),
                'strain': loss_strain.item(),
                'bc_left': loss_bc_left.item(),
                'bc_right': loss_bc_right.item(),
                'traction': loss_traction.item(),
                'rf': loss_rf.item(),
                'pde': loss_pde.item(),
                'penalty': loss_param_penalty.item()  # Track penalty to diagnose initialization spikes
            })

            # Store current weights for tracking
            self.weight_history.append({
                'data': self.lambda_data,
                'pde': self.lambda_pde,
                'bc_left': self.lambda_bc_left,
                'bc_right': self.lambda_bc_right,
                'rf': self.lambda_rf
            })

            # Store parameter values
            # Material params history (NEW: stiffness components)
            C11_inf_val = self.mat.C11_inf.item()
            C12_inf_val = self.mat.C12_inf.item()
            C22_inf_val = self.mat.C22_inf.item()
            C23_inf_val = self.mat.C23_inf.item()
            C44_inf_val = self.mat.C44_inf.item()
            C66_inf_val = self.mat.C66_inf.item()

            # Get spectrum weights
            spectrum = self.mat.get_spectrum_weights()

            self.param_history.append({
                # Instantaneous stiffness C(0)
                'C11_inst': self.mat.C11_inst.item(),
                'C12_inst': self.mat.C12_inst.item(),
                'C22_inst': self.mat.C22_inst.item(),
                'C23_inst': self.mat.C23_inst.item(),
                'C44_inst': self.mat.C44_inst.item(),
                'C66_inst': self.mat.C66_inst.item(),
                # Long-term stiffness C^∞
                'C11_inf': C11_inf_val,
                'C12_inf': C12_inf_val,
                'C22_inf': C22_inf_val,
                'C23_inf': C23_inf_val,
                'C44_inf': C44_inf_val,
                'C66_inf': C66_inf_val,
                # Spectrum weights (first 3 as examples)
                'g0': spectrum[0].item(),
                'g1': spectrum[1].item(),
                'g2': spectrum[2].item(),
                'g3': spectrum[3].item(),
                'g4': spectrum[4].item(),
                'A_g': (spectrum[2] + spectrum[3] + spectrum[4]).item(),
                'g_inf': spectrum[-1].item()
            })

            if epoch % log_interval == 0 or epoch == epochs - 1:
                weighted_data = self.lambda_data * loss_data.item()
                weighted_bc_left = self.lambda_bc_left * loss_bc_left.item()
                weighted_bc_right = self.lambda_bc_right * loss_bc_right.item()
                weighted_traction = self.lambda_traction * loss_traction.item()
                weighted_rf = self.lambda_rf * loss_rf.item()
                weighted_pde = self.lambda_pde * loss_pde.item()
                weighted_penalty = 50.0 * loss_param_penalty.item()

                # CRITICAL: Show penalty in first 500 epochs to diagnose initialization spikes
                # Epoch 0 total>>1 is usually penalty-dominated, not training collapse
                weighted_strain = self.lambda_strain * loss_strain.item()
                if epoch < 500:
                    print(
                        f"[Epoch {epoch}] "
                        f"total={loss_total.item():.4e} | "
                        f"data={loss_data.item():.4e} (w={weighted_data:.4e}), "
                        f"strain={loss_strain.item():.4e} (w={weighted_strain:.4e}), "
                        f"bcL={loss_bc_left.item():.4e} (w={weighted_bc_left:.4e}), "
                        f"bcR={loss_bc_right.item():.4e} (w={weighted_bc_right:.4e}), "
                        f"trac={loss_traction.item():.4e} (w={weighted_traction:.4e}), "
                        f"rf={loss_rf.item():.4e} (w={weighted_rf:.4e}), "
                        f"pde={loss_pde.item():.4e} (w={weighted_pde:.4e}), "
                        f"penalty={loss_param_penalty.item():.4e} (w={weighted_penalty:.4e})"
                    )
                else:
                    # After 500 epochs, hide penalty to keep logs concise
                    print(
                        f"[Epoch {epoch}] "
                        f"total={loss_total.item():.4e} | "
                        f"data={loss_data.item():.4e} (w={weighted_data:.4e}), "
                        f"strain={loss_strain.item():.4e} (w={weighted_strain:.4e}), "
                        f"bcL={loss_bc_left.item():.4e} (w={weighted_bc_left:.4e}), "
                        f"bcR={loss_bc_right.item():.4e} (w={weighted_bc_right:.4e}), "
                        f"trac={loss_traction.item():.4e} (w={weighted_traction:.4e}), "
                        f"rf={loss_rf.item():.4e} (w={weighted_rf:.4e}), "
                        f"pde={loss_pde.item():.4e} (w={weighted_pde:.4e})"
                    )

                print(f"Epoch {epoch}/{epochs} | Loss={loss_total.item():.3e} | "
                      f"Scheme-B inverse identification")

                # Time window and spectrum info
                t_start = sampled_times[0]
                t_end = sampled_times[-1]
                print(f"  [TimeWindow] t=[{t_start:.3f}, {t_end:.3f}]s, N_t={N_t}")

                lr_net_now  = optimizer.param_groups[0]['lr']
                lr_elas_now = optimizer.param_groups[1]['lr']
                lr_spec_now = optimizer.param_groups[2]['lr']
                print(f"  [LR] net={lr_net_now:.2e}, elastic={lr_elas_now:.2e}, spec={lr_spec_now:.2e}")

                print(f"  [Elastic] "
                      f"C11={self.mat.C11_inst.item():.0f}(T:11250)  "
                      f"C22={self.mat.C22_inst.item():.0f}(T:1426)  "
                      f"C12={self.mat.C12_inst.item():.0f}(F:891)  "
                      f"C23={self.mat.C23_inst.item():.0f}(F:916)  "
                      f"C66={self.mat.C66_inst.item():.0f}(T:268)")

                # Full Prony spectrum (all 7 weights, not just g0 and g_inf)
                tau_vals = self.mat.rho.cpu().tolist()
                g_str = "  ".join(
                    f"g{k}(τ={tau_vals[k]:.0f}s)={spectrum[k].item():.4f}"
                    for k in range(self.mat.N_prony)
                )
                A_g_val = (spectrum[2] + spectrum[3] + spectrum[4]).item()
                print(f"  [Spectrum] {g_str}  g_inf={spectrum[-1].item():.4f}  A_g={A_g_val:.4f}")

        print(f"Complete: {time.time() - start_time:.1f}s")

    def evaluate_rf_curve(self, fe_loader):
        """
        Evaluate RF(t) on the exact right-face nodes (same as data extraction).
        Uses recursive q on the face, without random sampling.
        Returns: times (list), rf_pred (list)
        """
        x_max = fe_loader.full_data['X'].max()
        right_nodes = fe_loader.full_data[np.abs(fe_loader.full_data['X'] - x_max) < 1e-4]
        frames = sorted(right_nodes['Frame'].unique())
        if len(frames) < 2:
            raise ValueError("Not enough time steps for RF evaluation.")

        strain_global_list = []
        strain_mat_list = []
        times = []  # Track corresponding times for dt computation

        # Collect strain history on right face
        for frame_idx in frames:
            slice_t = right_nodes[right_nodes['Frame'] == frame_idx]
            t_val = slice_t['Time'].iloc[0]  # Get corresponding time
            times.append(t_val)
            coords = torch.tensor(slice_t[['X', 'Y', 'Z']].values,
                                  dtype=torch.float32, device=device)
            t_tensor = torch.full((coords.shape[0], 1), t_val, dtype=torch.float32, device=device)

            # CRITICAL FIX: coords must require_grad BEFORE normalization
            # Otherwise u_pred won't depend on coords, breaking strain gradient computation
            coords_x = coords[:, 0:1].clone().requires_grad_(True)
            coords_y = coords[:, 1:2].clone().requires_grad_(True)
            coords_z = coords[:, 2:3].clone().requires_grad_(True)

            # Normalize coordinates (using grad-enabled coords)
            xh = (coords_x - self.x_min) / self.Lx
            yh = (coords_y - self.y_min) / self.Ly
            zh = (coords_z - self.z_min) / self.Lz
            th = (t_tensor - self.t_min) / self.T_max

            u_pred = self.model(xh, yh, zh, th)
            strain_global = self.compute_strain(u_pred, coords_x, coords_y, coords_z)
            strain_global_list.append(strain_global.detach())
            strain_mat_list.append(self.to_material_coords(strain_global).detach())

        strain_mat_seq = torch.stack(strain_mat_list, dim=0)
        dt_seq = torch.tensor(np.diff(times), dtype=torch.float32, device=device)
        q_seq = self.compute_q_recursive(strain_mat_seq, dt_seq, self.mat.rho)

        rf_preds = []
        area = (self.y_max - self.y_min) * (self.z_max - self.z_min)
        for i, t_val in enumerate(times):
            strain_g = strain_global_list[i]
            q_t = q_seq[i]
            stress_g = self.compute_stress(strain_g, q_t)
            sigma_11 = stress_g[:, 0]
            rf_pred = sigma_11.mean() * area
            rf_preds.append(rf_pred.item())

        return times, rf_preds

    def save_loss_history(self, filename):
        """Save loss history to CSV"""
        df = pd.DataFrame({
            'loss_total': self.loss_history,
            'loss_data': [h['data'] for h in self.loss_components_history],
            'loss_strain': [h.get('strain', 0.0) for h in self.loss_components_history],
            'loss_bc_left': [h['bc_left'] for h in self.loss_components_history],
            'loss_bc_right': [h['bc_right'] for h in self.loss_components_history],
            'loss_traction': [h['traction'] for h in self.loss_components_history],
            'loss_rf': [h['rf'] for h in self.loss_components_history],
            'loss_pde': [h['pde'] for h in self.loss_components_history]
        })
        df.to_csv(filename, index=False)
        print(f"Loss history saved to {filename}")

    def save_param_history(self, filename):
        """
        Save parameter evolution history with SHARED-SPECTRUM structure.

        Exports:
        - C_ij(0): Instantaneous stiffness
        - C_ij^∞: Long-term stiffness
        - g_k: Spectrum weights (first 3 + g_inf)
        """
        data = {
            # Instantaneous stiffness C(0)
            'C11_inst': [h['C11_inst'] for h in self.param_history],
            'C12_inst': [h['C12_inst'] for h in self.param_history],
            'C22_inst': [h['C22_inst'] for h in self.param_history],
            'C23_inst': [h['C23_inst'] for h in self.param_history],
            'C44_inst': [h['C44_inst'] for h in self.param_history],
            'C66_inst': [h['C66_inst'] for h in self.param_history],
            # Long-term stiffness (C^∞)
            'C11_inf': [h['C11_inf'] for h in self.param_history],
            'C12_inf': [h['C12_inf'] for h in self.param_history],
            'C22_inf': [h['C22_inf'] for h in self.param_history],
            'C23_inf': [h['C23_inf'] for h in self.param_history],
            'C44_inf': [h['C44_inf'] for h in self.param_history],
            'C66_inf': [h['C66_inf'] for h in self.param_history],
            # Spectrum weights
            'g0': [h['g0'] for h in self.param_history],
            'g1': [h['g1'] for h in self.param_history],
            'g2': [h['g2'] for h in self.param_history],
            'g3': [h['g3'] for h in self.param_history],
            'g4': [h['g4'] for h in self.param_history],
            'A_g': [h['A_g'] for h in self.param_history],
            'g_inf': [h['g_inf'] for h in self.param_history],
        }

        df = pd.DataFrame(data)
        df.to_csv(filename, index=False)
        print(f"Parameter history saved to {filename}")
        print(f"  Columns: C_ij(0), C_ij^∞, g0/g1 fixed, g2/g3/g4, A_g, g_inf")
        print(f"  Model: Scheme B reduced inverse identification")

    def save_weight_history(self, filename):
        """Save adaptive weight evolution history to CSV"""
        if not self.weight_history:
            print("No loss-weight history to save.")
            return

        df = pd.DataFrame({
            'lambda_data': [h['data'] for h in self.weight_history],
            'lambda_pde': [h['pde'] for h in self.weight_history],
            'lambda_bc_left': [h['bc_left'] for h in self.weight_history],
            'lambda_bc_right': [h['bc_right'] for h in self.weight_history],
            'lambda_rf': [h['rf'] for h in self.weight_history]
        })
        df.to_csv(filename, index=False)
        print(f"Loss-weight history saved to {filename}")


# 5. Visualization Functions

def plot_training_history(solver, save_dir):
    """Plot training loss curves"""
    plt.rcParams['font.family'] = 'Times New Roman'
    plt.rcParams['font.size'] = 12

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Total loss
    axes[0].semilogy(solver.loss_history, 'b-', linewidth=2)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Total Loss')
    axes[0].set_title('Total Loss Evolution')
    axes[0].grid(True, alpha=0.3)

    # Loss components
    epochs = range(len(solver.loss_history))
    axes[1].semilogy(epochs, [h['data'] for h in solver.loss_components_history],
                    label='Data', linewidth=2)
    axes[1].semilogy(epochs, [h['bc_left'] for h in solver.loss_components_history],
                    label='BC Left', linewidth=2)
    axes[1].semilogy(epochs, [h['bc_right'] for h in solver.loss_components_history],
                    label='BC Right', linewidth=2, linestyle='--')
    axes[1].semilogy(epochs, [h['rf'] for h in solver.loss_components_history],
                    label='RF', linewidth=2)
    axes[1].semilogy(epochs, [h['pde'] for h in solver.loss_components_history],
                    label='PDE', linewidth=2, alpha=0.7)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss Components')
    axes[1].set_title('Loss Components')
    axes[1].legend(fontsize=8)
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    save_path = Path(save_dir) / 'training_loss_history.png'
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"Training history plot saved to {save_path}")
    plt.close()

def plot_parameter_evolution(solver, save_dir):
    """
    Plot evolution of SHARED-SPECTRUM Prony model parameters.

    Shows:
    - Instantaneous C(0) vs Long-term C^∞
    - Relaxation strengths ΔC = C(0) - C^∞
    - Spectrum weights evolution
    """
    plt.rcParams['font.family'] = 'Times New Roman'
    plt.rcParams['font.size'] = 12

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    epochs = range(len(solver.param_history))

    # Panel 1: C66 instantaneous vs long-term (C11 is fixed, show C66 instead)
    ax = axes[0, 0]
    ax.plot(epochs, [h['C66_inst'] for h in solver.param_history],
            'g-', linewidth=2, label='C66(0) Instantaneous')
    ax.plot(epochs, [h['C66_inf'] for h in solver.param_history],
            'g--', linewidth=2, label='C66^∞ Long-term')
    ax.axhline(y=267.69, color='k', linestyle=':', alpha=0.5, label='True C66(0)≈267.7')
    ax.axhline(y=0.53, color='k', linestyle=':', alpha=0.3, label='True C66^∞≈0.53')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('C66 (MPa)')
    ax.set_title('C66: Instantaneous vs Long-term (Shear)')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    # Panel 2: C22 instantaneous vs long-term
    ax = axes[0, 1]
    ax.plot(epochs, [h['C22_inst'] for h in solver.param_history],
            'b-', linewidth=2, label='C22(0) Instantaneous')
    ax.plot(epochs, [h['C22_inf'] for h in solver.param_history],
            'b--', linewidth=2, label='C22^∞ Long-term')
    ax.axhline(y=1425.85, color='k', linestyle=':', alpha=0.5, label='True C22(0)≈1425.9')
    ax.axhline(y=2.84, color='k', linestyle=':', alpha=0.3, label='True C22^∞≈2.84')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('C22 (MPa)')
    ax.set_title('C22: Instantaneous vs Long-term (Transverse)')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    # Panel 3: Relaxation strengths ΔC = C(0) - C^∞
    ax = axes[1, 0]
    dC22 = [h['C22_inst'] - h['C22_inf'] for h in solver.param_history]
    dC23 = [h['C23_inst'] - h['C23_inf'] for h in solver.param_history]
    dC66 = [h['C66_inst'] - h['C66_inf'] for h in solver.param_history]

    ax.plot(epochs, dC22, 'b-', linewidth=2, label='ΔC22 (True≈1423)')
    ax.plot(epochs, dC23, 'm-', linewidth=2, label='ΔC23')
    ax.plot(epochs, dC66, 'g-', linewidth=2, label='ΔC66 (True≈267)')
    ax.axhline(y=0, color='k', linestyle='--', alpha=0.3)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('ΔC = C(0) - C^∞ (MPa)')
    ax.set_title('Relaxation Strengths')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Panel 4: Spectrum evolution
    ax = axes[1, 1]
    ax.plot(epochs, [h['g0'] for h in solver.param_history],
            'r-', linewidth=2, label='g₀ (ρ=0.1s)')
    ax.plot(epochs, [h['g1'] for h in solver.param_history],
            'b-', linewidth=2, label='g₁ (ρ=1s)')
    ax.plot(epochs, [h['g2'] for h in solver.param_history],
            'g-', linewidth=2, label='g₂ (ρ=10s)')
    ax.plot(epochs, [h['g_inf'] for h in solver.param_history],
            'k--', linewidth=2, label='g_∞ (equilibrium)')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Weight (normalized, Σg=1)')
    ax.set_title('Scheme B Spectrum Weights')
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    save_path = Path(save_dir) / 'parameter_evolution.png'
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"Parameter evolution plot saved to {save_path}")
    plt.close()

def plot_weight_evolution(solver, save_dir):
    """Plot loss weights recorded during training."""
    if not solver.weight_history:
        print("No loss-weight history available.")
        return

    plt.rcParams['font.family'] = 'Times New Roman'
    plt.rcParams['font.size'] = 12

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    n_epochs = len(solver.weight_history)
    epochs = range(n_epochs)
    # RF and BC_right weights
    ax = axes[0, 0]
    ax.semilogy(epochs, [h['rf'] for h in solver.weight_history],
               'r-', linewidth=2, label='λ_rf')
    ax.semilogy(epochs, [h['bc_right'] for h in solver.weight_history],
               'b-', linewidth=2, label='λ_bc_right')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Weight (log scale)')
    ax.set_title('RF and Right BC Weights')
    ax.legend()
    ax.grid(True, alpha=0.3, which='both')

    # PDE and BC_left weights
    ax = axes[0, 1]
    ax.semilogy(epochs, [h['pde'] for h in solver.weight_history],
               'g-', linewidth=2, label='λ_pde')
    ax.semilogy(epochs, [h['bc_left'] for h in solver.weight_history],
               'purple', linewidth=2, label='λ_bc_left')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Weight (log scale)')
    ax.set_title('PDE and Left BC Weights')
    ax.legend()
    ax.grid(True, alpha=0.3, which='both')

    # Data weight
    ax = axes[1, 0]
    ax.plot(epochs, [h['data'] for h in solver.weight_history],
           'orange', linewidth=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('λ_data')
    ax.set_title('Data Loss Weight')
    ax.grid(True, alpha=0.3)

    # Normalized weight distribution
    ax = axes[1, 1]
    rf_weights = np.array([h['rf'] for h in solver.weight_history])
    bc_right_weights = np.array([h['bc_right'] for h in solver.weight_history])
    bc_left_weights = np.array([h['bc_left'] for h in solver.weight_history])
    pde_weights = np.array([h['pde'] for h in solver.weight_history])
    data_weights = np.array([h['data'] for h in solver.weight_history])

    total = rf_weights + bc_right_weights + bc_left_weights + pde_weights + data_weights

    ax.plot(epochs, rf_weights / total, 'r-', linewidth=2, label='RF')
    ax.plot(epochs, bc_right_weights / total, 'b-', linewidth=2, label='BC_right')
    ax.plot(epochs, bc_left_weights / total, 'purple', linewidth=2, label='BC_left')
    ax.plot(epochs, pde_weights / total, 'g-', linewidth=2, label='PDE')
    ax.plot(epochs, data_weights / total, 'orange', linewidth=2, label='Data')

    ax.set_xlabel('Epoch')
    ax.set_ylabel('Relative Weight (fraction)')
    ax.set_title('Normalized Weight Distribution')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    save_path = Path(save_dir) / 'weight_evolution.png'
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"Weight evolution plot saved to {save_path}")
    plt.close()


## Training Configuration


In [ ]:
# RF loss control
use_rf_loss = True
lambda_rf = 5.0 if use_rf_loss else 0.0

# Remaining loss weights (aligned with EX2 setting)
lambda_data = 10.0
lambda_strain = 10.0
lambda_pde = 5.0
lambda_bc_left = 1.0
lambda_bc_right = 10.0
lambda_traction = 5.0

# Training hyperparameters
epochs = 10000
batch_size = 48
lr_network = 2e-3
lr_elastic = 1e-3
lr_params = 1e-3
log_interval = 100
seq_length_efficient = 24

print(f'use_rf_loss={use_rf_loss}, lambda_rf={lambda_rf}')


## Training and Saving Results


In [ ]:
script_dir = Path('.')
data_dir = script_dir / 'EX2-RESULTS'
rf_file = script_dir / 'ex-2-StressRelaxation-RF.csv'
u_file = script_dir / 'ex-2-StressRelaxation-U.csv'
frame_time_file = script_dir / 'frame-time.csv'

fe_loader = FEDataLoader(data_dir, rf_file, u_file, frame_time_file)
fe_loader.load_all_data()
bounds = fe_loader.get_domain_bounds()

mat = InverseMaterialParams().to(device)
model = LSTMInversePINN(
    spatial_hidden=(64, 64),
    lstm_hidden=64,
    lstm_layers=1,
    dropout=0.0,
).to(device)

solver = InverseLSTMPINNSolver(
    model, mat, fe_loader, bounds,
    lambda_data=lambda_data,
    lambda_strain=lambda_strain,
    lambda_pde=lambda_pde,
    lambda_bc_left=lambda_bc_left,
    lambda_bc_right=lambda_bc_right,
    lambda_traction=lambda_traction,
    lambda_rf=lambda_rf,
)

solver.train(
    epochs=epochs,
    batch_size=batch_size,
    lr_network=lr_network,
    lr_elastic=lr_elastic,
    lr_params=lr_params,
    log_interval=log_interval,
    seq_length_efficient=seq_length_efficient,
)

tag = 'rf' if lambda_rf > 0 else 'norf'
loss_csv = script_dir / f'nb_ex2_lstm_{tag}_training_loss_history.csv'
param_csv = script_dir / f'nb_ex2_lstm_{tag}_parameter_history.csv'
weight_csv = script_dir / f'nb_ex2_lstm_{tag}_weight_history.csv'
model_path = script_dir / f'nb_ex2_lstm_{tag}_model.pth'
mat_path = script_dir / f'nb_ex2_lstm_{tag}_mat_params.pth'

solver.save_loss_history(loss_csv)
solver.save_param_history(param_csv)
solver.save_weight_history(weight_csv)
plot_training_history(solver, script_dir)
plot_parameter_evolution(solver, script_dir)
plot_weight_evolution(solver, script_dir)

torch.save(model.state_dict(), model_path)
torch.save(mat.state_dict(), mat_path)

print('Saved:')
for p in [loss_csv, param_csv, weight_csv, model_path, mat_path]:
    print(' ', p)
